# Understanding Support Vector Machines from First Principles
### Margin, Regularisation, Kernels and Support-Vector Composition in Practice

**Student:** Teddy Ryu (12108700)

This public notebook is the executable entry point for the complete project. The from-scratch implementations are maintained as modular Python files in the linked public GitHub repository and are loaded below so the same source is used by both the notebook and the reproducible command-line workflow.

## 1. Environment and source files

In [ ]:
!pip -q install numpy scikit-learn matplotlib
!git clone -q https://github.com/TeddybearAi/svm-from-first-principles.git
%cd /content/svm-from-first-principles
print('Repository ready.')

The implementation is split into five auditable modules:

- `01_data.py`: data generation, real-data loading, splitting and standardisation
- `02_primal_svm.py`: primal soft-margin SVM from scratch
- `03_dual_smo.py`: simplified dual SMO SVM from scratch
- `04_experiments.py`: validation checks, Sweeps A–F and verification diagnostics
- `05_make_figures.py`: generation of Figures 1–5

In [ ]:
from pathlib import Path
for fn in ['01_data.py','02_primal_svm.py','03_dual_smo.py','04_experiments.py','05_make_figures.py']:
    print(f'{fn}: {len(Path(fn).read_text().splitlines())} lines')

## 2. Load the from-scratch implementations

In [ ]:
import importlib.util
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

data01 = load_module('data01', '01_data.py')
primal02 = load_module('primal02', '02_primal_svm.py')
dual03 = load_module('dual03', '03_dual_smo.py')
exp04 = load_module('exp04', '04_experiments.py')
print('Loaded PrimalSVM and DualSVM_SMO from the public source files.')

## 3. Basic implementation checks

In [ ]:
import numpy as np
from sklearn.svm import SVC

X, y = data01.make_synthetic(overlap='low')
(Xtr, ytr), (Xval, yval), (Xte, yte), _ = data01.split_and_scale(X, y)
m = primal02.PrimalSVM(C=1.0, lr=0.01, n_epochs=3000).fit(Xtr, ytr)
ref = SVC(kernel='linear', C=1.0).fit(Xtr, ytr)
print('Primal scratch test accuracy:', (m.predict(Xte) == yte).mean())
print('sklearn reference accuracy:', ref.score(Xte, yte))
primal02.verify_against_sklearn(m, ref, Xte, label='Synthetic low-overlap')

## 4. C=100 geometry spot checks used in the report

In [ ]:
c100_checks = exp04.primal_c100_spot_checks()
for r in c100_checks:
    print(f"{r['overlap']} overlap: scratch ||w||={r['scratch_w_norm']:.6f}, sklearn ||w||={r['sklearn_w_norm']:.6f}, cosine={r['cosine_similarity']:.9f}, test acc={r['scratch_test_acc']:.3f}/{r['sklearn_test_acc']:.3f}")

## 5. Full experimental sweeps

The next cell regenerates the full experiment set reported in the paper: validation of implementation settings, Sweeps A–F, KKT/tolerance diagnostics, the paired real-data comparison, and the failure-case analysis. The simplified SMO sweeps can take some time on Colab CPU.

In [ ]:
!python 04_experiments.py

## 6. Figures 1–5

In [ ]:
!python 05_make_figures.py
from IPython.display import display, Image
for fn in ['fig1_primal_C_sweep.png','fig2_dual_sv_counts.png','fig3_kernel_comparison_moons.png','fig4_real_dataset_C_sweep.png','fig5_sv_composition.png']:
    display(Image(filename=fn))

## 7. Raw results

The full numerical outputs used for the report tables are saved to `sweep_results.json`. The separate `c100_primal_verification.json` records the explicit C=100 scratch-vs-sklearn geometry checks.